# 03 - Exploratory Data Analysis (EDA) & Statistical Analysis
**Project:** DS Job Recommend (Job Level Prediction & Recommendation System)  
**Assigned Role:** **PERSON 2 — Branch:** `feature/visualization`  
**Input Dataset:** `data/processed/postings_clean.csv` (123,849 rows × 31 columns)  
**Target Variable:** `formatted_experience_level`

---
### TABLE OF CONTENTS
1. **Environment Setup & Data Loading**
2. **Univariate Analysis** (Numerical & Categorical)
3. **Feature ↔ Feature Analysis** (Correlation & Multicollinearity)
4. **Feature ↔ Target Analysis** (Group Comparison & Statistical Testing)
5. **Categorical ↔ Numerical Analysis**
6. **Categorical ↔ Categorical Analysis** (Crosstab & Chi-Square Test)
7. **Multivariate Analysis** (Interaction & Full Heatmap)
8. **Outlier Analysis & Log Transformation**
9. **Class Imbalance Analysis & Modeling Recommendations**
10. **Skill Analysis**
11. **Text Analysis**
12. **Business Insights & Executive Dashboard**
---

## 1. ENVIRONMENT SETUP & DATA LOADING
**CELL 1: Environment Setup & Library Imports**

In [ ]:
import os
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from scipy.stats import chi2_contingency, f_oneway, kruskal

# Tắt cảnh báo không cần thiết
warnings.filterwarnings('ignore')

# Cài đặt hiển thị dữ liệu
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# Cài đặt giao diện đồ thị đồng bộ
sns.set_theme(style='whitegrid')
sns.set_palette('Set2')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# Thêm thư mục src vào sys.path để có thể import module dùng chung
ROOT_DIR = Path(__file__).resolve().parent.parent if '__file__' in locals() else Path.cwd().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

# Tự động nhận diện đường dẫn thư mục data/processed
if (ROOT_DIR / 'data' / 'processed').exists():
    PROCESSED = ROOT_DIR / 'data' / 'processed'
elif Path('../data/processed').exists():
    PROCESSED = Path('../data/processed')
else:
    PROCESSED = Path('data/processed')

print(f'Setup completed successfully. Data directory: {PROCESSED}')

**CELL 2: Load Processed Datasets**

In [ ]:
# 1. Đọc bảng chính: postings_clean.csv
df = pd.read_csv(PROCESSED / 'postings_clean.csv', low_memory=False)
print(f'Main table (postings_clean): {df.shape[0]:,} rows x {df.shape[1]} columns')

# 2. Hàm đọc các bảng phụ trợ an toàn
def load_auxiliary_table(filename):
    p = PROCESSED / filename
    if p.exists():
        data = pd.read_csv(p)
        print(f'-> Loaded {filename}: {data.shape[0]:,} rows x {data.shape[1]} columns')
        return data
    print(f'-> File not found: {filename}')
    return None

job_skills     = load_auxiliary_table('job_skills_clean.csv')
companies      = load_auxiliary_table('companies_clean.csv')
salaries       = load_auxiliary_table('salaries_clean.csv')
job_industries = load_auxiliary_table('job_industries_clean.csv')

**CELL 3: Feature Grouping & Missing Value Analysis**

In [ ]:
# Target variable
TARGET = 'formatted_experience_level'

# Numerical features
NUM_COLS = [c for c in ['min_salary', 'med_salary', 'max_salary',
                         'normalized_salary', 'views', 'applies']
            if c in df.columns]

# Categorical features
CAT_COLS = [c for c in ['formatted_work_type', 'work_type', 'remote_allowed',
                          'pay_period', 'compensation_type', 'currency']
            if c in df.columns]

# Text features
TEXT_COLS = [c for c in ['title', 'description', 'skills_desc'] if c in df.columns]

print('=== FEATURE GROUPING SUMMARY ===')
print(f'Target Variable      : {TARGET}')
print(f'Missing Target Rows  : {df[TARGET].isnull().sum():,} ({df[TARGET].isnull().mean()*100:.2f}%)')
print(f'Numerical Features   : {NUM_COLS}')
print(f'Categorical Features : {CAT_COLS}')
print(f'Text Features        : {TEXT_COLS}\n')

# Missing value summary
missing_df = pd.DataFrame({
    'Data Type'        : df.dtypes,
    'Missing Count'    : df.isnull().sum(),
    'Missing Ratio (%)': (df.isnull().sum() / len(df) * 100).round(2),
    'Unique Values'    : df.nunique()
})
print('=== TOP 10 COLUMNS WITH HIGHEST MISSING RATIO ===')
missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Ratio (%)', ascending=False).head(10)

## 2. UNIVARIATE ANALYSIS
Analyze individual feature distributions: Mean, Median, Std, IQR, Skewness, Kurtosis, and Outliers.

---
**CELL 4: Numerical Descriptive Statistics**

In [ ]:
print('=== NUMERICAL DESCRIPTIVE STATISTICS ===')
desc = df[NUM_COLS].describe().T
desc['IQR']      = desc['75%'] - desc['25%']
desc['Skewness'] = df[NUM_COLS].skew()
desc['Kurtosis'] = df[NUM_COLS].kurtosis()
desc.rename(columns={'25%': 'Q1', '50%': 'Median', '75%': 'Q3'}, inplace=True)
desc.round(2)

**CELL 5: Numerical Visualization (Histogram, KDE & Boxplot)**

In [ ]:
for col in NUM_COLS:
    data = df[col].dropna()
    if len(data) == 0:
        continue

    # IQR calculation for outliers
    Q1, Q3 = data.quantile(0.25), data.quantile(0.75)
    IQR    = Q3 - Q1
    lower  = Q1 - 1.5 * IQR
    upper  = Q3 + 1.5 * IQR
    n_out  = ((data < lower) | (data > upper)).sum()

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # 1. Histogram + KDE
    axes[0].hist(data, bins=60, color='steelblue', edgecolor='white', alpha=0.7, density=True)
    data.plot.kde(ax=axes[0], color='crimson', linewidth=2)
    axes[0].axvline(data.mean(),   color='orange', linestyle='--', linewidth=1.5,
                    label=f'Mean = {data.mean():,.0f}')
    axes[0].axvline(data.median(), color='green',  linestyle='--', linewidth=1.5,
                    label=f'Median = {data.median():,.0f}')
    axes[0].set_title(f'Distribution (Histogram & KDE) — {col}', fontsize=12, fontweight='bold')
    axes[0].legend()

    # Skewness annotation
    skew = data.skew()
    txt  = 'Right-Skewed' if skew > 0.5 else ('Left-Skewed' if skew < -0.5 else 'Symmetric')
    axes[0].text(0.97, 0.95, f'Skew = {skew:.2f}\n({txt})',
                 transform=axes[0].transAxes, ha='right', va='top', fontsize=9,
                 bbox=dict(facecolor='lightyellow', alpha=0.9, boxstyle='round'))

    # 2. Boxplot
    axes[1].boxplot(data, vert=False, patch_artist=True,
                    boxprops=dict(facecolor='steelblue', alpha=0.6),
                    medianprops=dict(color='red', linewidth=2),
                    flierprops=dict(marker='.', color='gray', alpha=0.3, markersize=3))
    axes[1].set_title(f'Boxplot — {col}', fontsize=12, fontweight='bold')
    axes[1].text(0.97, 0.85,
                 f'Q1 = {Q1:,.0f}\nQ3 = {Q3:,.0f}\nIQR = {IQR:,.0f}\n'
                 f'Outliers: {n_out:,} ({n_out/len(data)*100:.1f}%)',
                 transform=axes[1].transAxes, ha='right', va='top', fontsize=9,
                 bbox=dict(facecolor='lightyellow', alpha=0.9, boxstyle='round'))

    plt.suptitle(f'Univariate Analysis (Numerical): {col} (n={len(data):,}, Missing={df[col].isnull().sum():,})',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

**CELL 6: Categorical Frequency & Percentage Distribution**

In [ ]:
for col in CAT_COLS:
    vc  = df[col].value_counts(dropna=False)
    pct = df[col].value_counts(normalize=True, dropna=False) * 100
    lbls = [str(x) if pd.notna(x) else '(Missing)' for x in vc.index]
    tbl = pd.DataFrame({'Category': lbls, 'Count': vc.values, 'Percentage (%)': pct.round(2).values})
    print(f'\nFeature: {col} | Cardinality={df[col].nunique()} | Missing={df[col].isnull().sum():,} ({df[col].isnull().mean()*100:.1f}%)')
    print(tbl.to_string(index=False))

**CELL 7: Categorical Visualization (Count & Percentage Charts)**

In [ ]:
for col in CAT_COLS:
    vc  = df[col].value_counts(dropna=False).head(15)
    pct = vc / len(df) * 100
    x_labels = [str(x) if pd.notna(x) else '(Missing)' for x in vc.index]
    colors   = sns.color_palette('Set2', len(vc))

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # Count bar chart
    bars = axes[0].bar(x_labels, vc.values, color=colors, edgecolor='white')
    for bar, p in zip(bars, pct.values):
        axes[0].text(bar.get_x() + bar.get_width() / 2,
                     bar.get_height() + vc.values.max() * 0.01,
                     f'{p:.1f}%', ha='center', va='bottom', fontsize=9)
    axes[0].set_title(f'Category Count — {col}', fontsize=12, fontweight='bold')
    axes[0].tick_params(axis='x', rotation=40)

    # Percentage bar chart
    axes[1].bar(x_labels, pct.values, color=colors, edgecolor='white')
    axes[1].set_ylabel('Percentage (%)')
    axes[1].set_title(f'Category Percentage — {col}', fontsize=12, fontweight='bold')
    axes[1].tick_params(axis='x', rotation=40)

    # Ideal balance baseline
    n_cat = df[col].nunique()
    if n_cat > 0:
        axes[1].axhline(100 / n_cat, color='red', linestyle='--', linewidth=1,
                        label=f'Uniform Baseline = {100/n_cat:.1f}%')
        axes[1].legend(fontsize=8)

    plt.suptitle(f'Univariate Analysis (Categorical): {col}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 3. FEATURE ↔ FEATURE ANALYSIS
Investigate pairwise relationships among numerical features and detect Multicollinearity.

---
**CELL 8: Pearson & Spearman Correlation Heatmap**

In [ ]:
# Compute Pearson (linear) and Spearman (rank-order) correlation matrices
pearson_corr  = df[NUM_COLS].corr(method='pearson')
spearman_corr = df[NUM_COLS].corr(method='spearman')

# Lower-triangle mask
mask = np.triu(np.ones_like(pearson_corr, dtype=bool))

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

sns.heatmap(pearson_corr, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, vmin=-1, vmax=1,
            ax=axes[0], linewidths=0.5, annot_kws={'size': 10})
axes[0].set_title('Pearson Correlation Matrix (Linear)', fontsize=13, fontweight='bold')

sns.heatmap(spearman_corr, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, vmin=-1, vmax=1,
            ax=axes[1], linewidths=0.5, annot_kws={'size': 10})
axes[1].set_title('Spearman Correlation Matrix (Monotonic)', fontsize=13, fontweight='bold')

plt.suptitle('Correlation Analysis Between Numerical Features (Numerical vs Numerical)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

**CELL 9: Correlation Classification & Multicollinearity Check**

In [ ]:
rows = []
for i in range(len(NUM_COLS)):
    for j in range(i + 1, len(NUM_COLS)):
        r  = pearson_corr.iloc[i, j]
        rs = spearman_corr.iloc[i, j]
        a  = abs(r)
        if a > 0.7:
            strength = 'Strong (>0.7)'
        elif a > 0.4:
            strength = 'Moderate (0.4–0.7)'
        else:
            strength = 'Weak (<0.4)'
        direction = 'Positive (+)' if r > 0 else 'Negative (−)'
        rows.append({
            'Feature 1'  : NUM_COLS[i],
            'Feature 2'  : NUM_COLS[j],
            'Pearson r'  : round(r, 3),
            'Spearman r' : round(rs, 3),
            'Strength'   : strength,
            'Direction'  : direction
        })

corr_df = pd.DataFrame(rows).sort_values('Pearson r', key=abs, ascending=False)
print('=== CORRELATION CLASSIFICATION SUMMARY ===')
print(corr_df.to_string(index=False))

# Multicollinearity check (|r| > 0.7)
print('\n=== MULTICOLLINEARITY ALERT (|r| > 0.7) ===')
multi = corr_df[corr_df['Pearson r'].abs() > 0.7]
if len(multi):
    print(multi[['Feature 1', 'Feature 2', 'Pearson r']].to_string(index=False))
    print('=> CONCLUSION: Salary columns carry virtually redundant information (r > 0.99).')
    print('=> HANDOVER TO PERSON 3 & 4: Keep only normalized_salary to avoid model instability.')
else:
    print('No severe multicollinearity detected.')

**CELL 10: Pair Plot Analysis**

In [ ]:
# Pair plot on key numerical features with substantial co-occurrence
pair_cols = [c for c in ['normalized_salary', 'views', 'applies'] if c in df.columns]
sample_df = df[pair_cols].dropna()

if len(sample_df) > 0:
    n_sample = min(2000, len(sample_df))
    sample = sample_df.sample(n_sample, random_state=42)
    g = sns.pairplot(sample, diag_kind='kde',
                     plot_kws={'alpha': 0.25, 's': 15, 'color': 'steelblue'},
                     diag_kws={'color': 'steelblue', 'fill': True})
    g.fig.suptitle(f'Pair Plot — Cross-Distribution of Numerical Features (Sample {n_sample:,} rows)',
                   fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

## 4. FEATURE ↔ TARGET ANALYSIS
Target: `formatted_experience_level` (Job Experience Level).  
Statistical hypothesis testing: ANOVA & Kruskal-Wallis.

---
**CELL 11: Numerical Statistics by Target (Job Level)**

In [ ]:
if TARGET in df.columns:
    for col in NUM_COLS:
        tbl = df.groupby(TARGET)[col].agg(['count', 'mean', 'median', 'std', 'min', 'max'])
        print(f'\n=== {col.upper()} STATISTICS BY JOB LEVEL ({TARGET}) ===')
        print(tbl.round(2))

**CELL 12: Numerical vs Target Visualization (Boxplot & Violin Plot)**

In [ ]:
if TARGET in df.columns:
    for col in NUM_COLS:
        plot_df = df[[TARGET, col]].dropna()
        if len(plot_df) == 0:
            continue

        # Sort Job Levels by median descending
        order = plot_df.groupby(TARGET)[col].median().sort_values(ascending=False).index

        fig, axes = plt.subplots(1, 2, figsize=(16, 5))

        # Boxplot
        sns.boxplot(data=plot_df, x=TARGET, y=col,
                    order=order, palette='Set2', ax=axes[0])
        axes[0].set_title(f'{col} by Job Level — Boxplot', fontsize=12, fontweight='bold')
        axes[0].tick_params(axis='x', rotation=30)

        # Violin Plot
        sns.violinplot(data=plot_df, x=TARGET, y=col, order=order,
                       palette='Set2', ax=axes[1], inner='quartile', cut=0)
        axes[1].set_title(f'{col} by Job Level — Violin Plot', fontsize=12, fontweight='bold')
        axes[1].tick_params(axis='x', rotation=30)

        plt.suptitle(f'Feature vs Target: {col} vs {TARGET}',
                     fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()

**CELL 13: Statistical Hypothesis Testing (ANOVA & Kruskal-Wallis)**

In [ ]:
if TARGET in df.columns:
    print('=== STATISTICAL HYPOTHESIS TESTING: ANOVA & KRUSKAL-WALLIS (α = 0.05) ===')
    print('H0: Numerical distribution is identical across all job levels.')
    print('H1: At least one job level exhibits a statistically significant difference.\n')

    stat_results = []
    for col in NUM_COLS:
        groups = [df[df[TARGET] == g][col].dropna().values
                  for g in df[TARGET].dropna().unique()]
        groups = [g for g in groups if len(g) > 1]
        if len(groups) < 2:
            continue

        f_val, p_anova = f_oneway(*groups)   # ANOVA (parametric)
        h_val, p_kw    = kruskal(*groups)    # Kruskal-Wallis (non-parametric)

        stat_results.append({
            'Feature'         : col,
            'ANOVA F'         : round(f_val, 2),
            'ANOVA p-value'   : round(p_anova, 5),
            'Kruskal-Wallis H': round(h_val, 2),
            'KW p-value'      : round(p_kw, 5),
            'Significant?'    : 'YES (p < 0.05)' if p_kw < 0.05 else 'NO'
        })

    print(pd.DataFrame(stat_results).to_string(index=False))
    print('\n=> INTERPRETATION: Kruskal-Wallis is preferred due to skewed salary and views distributions.')
    print('=> Both tests yield p < 0.001 -> Salary features possess strong discriminative power for Job Level!')

## 5. CATEGORICAL ↔ NUMERICAL ANALYSIS
Compare salary and engagement metrics (views, applies) across work types and remote policy.

---
**CELL 14: Salary & Interaction by Work Type & Remote Policy**

In [ ]:
sal_col = next((c for c in ['normalized_salary', 'med_salary', 'min_salary']
                if c in df.columns), None)

pairs = []
for cat in ['formatted_work_type', 'remote_allowed', 'pay_period']:
    if cat in df.columns and sal_col:
        pairs.append((cat, sal_col))
for num in ['views', 'applies']:
    if 'formatted_work_type' in df.columns and num in df.columns:
        pairs.append(('formatted_work_type', num))

for cat_col, num_col in pairs:
    plot_df = df[[cat_col, num_col]].dropna()
    if len(plot_df) == 0:
        continue

    order = plot_df.groupby(cat_col)[num_col].median().sort_values(ascending=False).index

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # Boxplot
    sns.boxplot(data=plot_df, x=cat_col, y=num_col,
                order=order, palette='Set2', ax=axes[0])
    axes[0].set_title(f'{num_col} by {cat_col} — Boxplot', fontsize=11, fontweight='bold')
    axes[0].tick_params(axis='x', rotation=30)

    # Mean bar chart
    means = plot_df.groupby(cat_col)[num_col].mean().reindex(order)
    x_mean_labels = [str(x) for x in means.index]
    bars  = axes[1].bar(x_mean_labels, means.values,
                         color=sns.color_palette('Set2', len(means)), edgecolor='white')
    for bar, v in zip(bars, means.values):
        axes[1].text(bar.get_x() + bar.get_width() / 2,
                     bar.get_height(), f'{v:,.0f}',
                     ha='center', va='bottom', fontsize=9)
    axes[1].set_title(f'Mean {num_col} by {cat_col}', fontsize=11, fontweight='bold')
    axes[1].tick_params(axis='x', rotation=30)

    plt.suptitle(f'Categorical vs Numerical: {cat_col} × {num_col}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 6. CATEGORICAL ↔ CATEGORICAL ANALYSIS
Crosstabs, Chi-Square Independence Tests, and Stacked Bar Charts.

---
**CELL 15: Crosstab & Chi-Square Independence Test**

In [ ]:
if TARGET in df.columns:
    cat_list     = [c for c in ['formatted_work_type', 'remote_allowed', 'pay_period']
                    if c in df.columns]
    chi2_results = []

    for col2 in cat_list:
        plot_df = df[[TARGET, col2]].dropna()
        if len(plot_df) == 0:
            continue

        # Crosstab count and row percentage
        ct     = pd.crosstab(plot_df[TARGET], plot_df[col2])
        ct_pct = pd.crosstab(plot_df[TARGET], plot_df[col2], normalize='index') * 100

        print(f'\n=== CROSSTAB: {TARGET} × {col2} ===')
        print(ct)
        print('\nRow percentage (%):')
        print(ct_pct.round(1))

        # Chi-square test
        chi2, p, dof, _ = chi2_contingency(ct)
        chi2_results.append({
            'Variable Pair' : f'{TARGET} × {col2}',
            'Chi2 Statistic': round(chi2, 2),
            'p-value'       : round(p, 5),
            'Degrees Freedom': dof,
            'Significant?   ': 'YES (Associated)' if p < 0.05 else 'NO'
        })

        # Stacked bar chart
        ct_pct.plot(kind='bar', stacked=True, figsize=(12, 5),
                    colormap='Set2', edgecolor='white', linewidth=0.5)
        plt.title(f'Stacked Bar Chart: {TARGET} × {col2}\n(Chi2 = {chi2:.1f}, p = {p:.4f})',
                  fontsize=12, fontweight='bold')
        plt.ylabel('Percentage (%)')
        plt.xticks(rotation=30)
        plt.legend(title=col2, bbox_to_anchor=(1.05, 1))
        plt.tight_layout()
        plt.show()

    print('\n=== CHI-SQUARE TEST SUMMARY ===')
    print(pd.DataFrame(chi2_results).to_string(index=False))

## 7. MULTIVARIATE ANALYSIS
Multi-factor interactions: Salary × Job Level × Work Type and Full Correlation Heatmap.

---
**CELL 16: Three-Way Interaction: Salary × Job Level × Work Type**

In [ ]:
sal_col  = next((c for c in ['normalized_salary', 'med_salary'] if c in df.columns), None)
work_col = 'formatted_work_type'

if TARGET in df.columns and sal_col and work_col in df.columns:
    plot_df = df[[TARGET, sal_col, work_col]].dropna()
    order   = plot_df.groupby(TARGET)[sal_col].median().sort_values().index

    fig, ax = plt.subplots(figsize=(14, 6))
    sns.boxplot(data=plot_df, x=TARGET, y=sal_col,
                hue=work_col, order=order, palette='Set2', ax=ax)
    ax.set_title(f'Three-Way Interaction: {sal_col} × {TARGET} × {work_col}',
                 fontsize=13, fontweight='bold')
    ax.tick_params(axis='x', rotation=30)
    ax.legend(title='Work Type', bbox_to_anchor=(1.05, 1))
    plt.tight_layout()
    plt.show()

**CELL 17: Full Correlation Heatmap (Numerical + Encoded Categorical)**

In [ ]:
df_enc = df[NUM_COLS].copy()
for col in CAT_COLS:
    if col in df.columns and df[col].nunique() <= 10:
        df_enc[col + '_code'] = df[col].astype('category').cat.codes

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(df_enc.corr(), annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.3, ax=ax, annot_kws={'size': 8})
ax.set_title('Comprehensive Correlation Heatmap (Numerical + Categorical Codes)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. OUTLIER ANALYSIS & LOG TRANSFORMATION
Outlier quantification via IQR and Z-Score (3σ), and evaluation of Log(1+x) transformation.

---
**CELL 18: Outlier Detection via IQR & Z-Score (3σ)**

In [ ]:
print('=== OUTLIER ANALYSIS SUMMARY (IQR vs Z-SCORE) ===')
outlier_rows = []
for col in NUM_COLS:
    data = df[col].dropna()
    if len(data) == 0:
        continue

    # IQR method
    Q1, Q3  = data.quantile(0.25), data.quantile(0.75)
    IQR     = Q3 - Q1
    lower   = Q1 - 1.5 * IQR
    upper   = Q3 + 1.5 * IQR
    n_iqr   = ((data < lower) | (data > upper)).sum()

    # Z-score method (threshold = 3 standard deviations)
    n_z = (np.abs(stats.zscore(data)) > 3).sum()

    outlier_rows.append({
        'Feature'        : col,
        'Count (n)'      : len(data),
        'IQR Outliers'   : n_iqr,
        'IQR Ratio (%)'  : round(n_iqr / len(data) * 100, 2),
        'Z>3σ Outliers'  : n_z,
        'Z Ratio (%)'    : round(n_z / len(data) * 100, 2),
        'IQR Lower Bound': round(lower, 1),
        'IQR Upper Bound': round(upper, 1)
    })

print(pd.DataFrame(outlier_rows).to_string(index=False))
print('\n=> OUTLIER NATURE: Salary outliers represent genuine high-earning leadership roles (Director / Executive).')
print('   DO NOT REMOVE THEM!')
print('=> RECOMMENDATION: Apply Log(1 + x) transformation to normalize the right-skewed distribution.')

**CELL 19: Log-Transformation for Right-Skewed Salary Features**

In [ ]:
sal_col = next((c for c in ['normalized_salary', 'min_salary'] if c in df.columns), None)
if sal_col:
    data   = df[sal_col].dropna()
    Q1, Q3 = data.quantile(0.25), data.quantile(0.75)
    IQR    = Q3 - Q1
    lower  = Q1 - 1.5 * IQR
    upper  = Q3 + 1.5 * IQR
    n_out  = ((data < lower) | (data > upper)).sum()

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Boxplot
    axes[0].boxplot(data, vert=False, patch_artist=True,
                    boxprops=dict(facecolor='steelblue', alpha=0.6),
                    medianprops=dict(color='red', linewidth=2),
                    flierprops=dict(marker='.', color='gray', alpha=0.2, markersize=3))
    axes[0].set_title(f'Original Boxplot: {sal_col}\nOutliers = {n_out:,} ({n_out/len(data)*100:.1f}%)',
                      fontsize=11, fontweight='bold')

    # Original Distribution
    axes[1].hist(data, bins=60, color='steelblue', alpha=0.7, edgecolor='white', density=True)
    axes[1].set_title(f'Original Distribution\nSkewness = {data.skew():.2f}',
                      fontsize=11, fontweight='bold')

    # After Log(1 + x) Transformation
    log_data = np.log1p(data[data > 0])
    axes[2].hist(log_data, bins=60, color='seagreen', alpha=0.7, edgecolor='white', density=True)
    axes[2].set_title(f'After Log(1+x) Transform\nSkewness = {log_data.skew():.2f}',
                      fontsize=11, fontweight='bold')
    axes[2].set_xlabel('log(1 + salary)')

    plt.suptitle(f'Impact of Log-Transformation on Outliers: {sal_col}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 9. CLASS IMBALANCE ANALYSIS
Target: `formatted_experience_level` — Assess imbalance ratio and provide modeling strategies for Person 4.

---
**CELL 20: Target Class Imbalance Analysis & Modeling Recommendations**

In [ ]:
if TARGET in df.columns:
    counts = df[TARGET].value_counts(dropna=False)
    pcts   = df[TARGET].value_counts(normalize=True, dropna=False) * 100

    target_labels = [str(x) if pd.notna(x) else '(Missing / NaN)' for x in counts.index]

    print('=== TARGET CLASS DISTRIBUTION ===')
    tbl_imbalance = pd.DataFrame({'Class': target_labels, 'Count': counts.values, 'Percentage (%)': pcts.round(2).values})
    print(tbl_imbalance.to_string(index=False))

    # Calculate Imbalance Ratio
    valid_counts = df[TARGET].dropna().value_counts()
    if len(valid_counts) > 1:
        ratio = valid_counts.max() / valid_counts.min()
        print(f'\n=> IMBALANCE RATIO (Majority / Minority Class) = {ratio:.1f} : 1')
        if ratio > 5:
            print('=> ASSESSMENT: SEVERE CLASS IMBALANCE DETECTED!')
            print('=> HANDOVER TO PERSON 4 (Modeling):')
            print('   1. Avoid raw Accuracy metric; adopt F1-Macro or Weighted F1-Score.')
            print('   2. Employ Stratified K-Fold cross-validation.')
            print('   3. Use class weighting (class_weight="balanced") or SMOTE resampling.')

    colors = sns.color_palette('Set2', len(counts))
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Bar chart
    bars = axes[0].bar(target_labels, counts.values, color=colors, edgecolor='white')
    for bar, cnt, pct in zip(bars, counts.values, pcts.values):
        axes[0].text(bar.get_x() + bar.get_width() / 2,
                     bar.get_height() + counts.values.max() * 0.01,
                     f'{cnt:,}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=9)
    axes[0].set_title('Job Posting Count by Experience Level', fontsize=13, fontweight='bold')
    axes[0].tick_params(axis='x', rotation=30)

    # Pie chart
    axes[1].pie(counts.values,
                labels=[f'{lbl}\n({p:.1f}%)' for lbl, p in zip(target_labels, pcts.values)],
                colors=colors, startangle=90,
                wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
    axes[1].set_title('Class Proportions of Experience Level', fontsize=13, fontweight='bold')

    plt.suptitle('Target Class Imbalance Analysis  →  Handover to Person 4',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 10. SKILL ANALYSIS
Extract and analyze skills from `job_skills_clean.csv` grouped by Job Experience Level.

---
**CELL 21: Top Overall Skills & Characteristic Skills by Job Level**

In [ ]:
if job_skills is not None:
    skill_col = next((c for c in ['skill_abr', 'skill_name', 'skill']
                      if c in job_skills.columns), None)
    if skill_col:
        top20 = job_skills[skill_col].value_counts().head(20)
        skill_labels = [str(x) if pd.notna(x) else '(Missing)' for x in top20.index]

        fig, ax = plt.subplots(figsize=(14, 6))
        ax.bar(skill_labels, top20.values,
               color=sns.color_palette('Set2', len(top20)), edgecolor='white')
        ax.set_title('Top 20 Most Demanded Skills Overall (job_skills_clean)',
                     fontsize=13, fontweight='bold')
        ax.tick_params(axis='x', rotation=45)
        plt.tight_layout()
        plt.show()

        # Skills breakdown by Job Level
        if TARGET in df.columns:
            id_col = next((c for c in ['job_id', 'jobId']
                           if c in job_skills.columns and c in df.columns), None)
            if id_col:
                merged = job_skills.merge(
                    df[[id_col, TARGET]].dropna(), on=id_col, how='inner')
                print('\n=== TOP 5 CHARACTERISTIC SKILLS BY JOB LEVEL ===')
                for level in sorted(merged[TARGET].unique()):
                    top5 = merged[merged[TARGET] == level][skill_col].value_counts().head(5)
                    print(f'\n  • Job Level: {level}')
                    print(top5.to_string())

## 11. TEXT ANALYSIS
Examine title/description text length and extract top keywords to support Person 3 in NLP Feature Engineering.

---
**CELL 22: Text Length & Word Count Statistics by Job Level**

In [ ]:
# Text length features
if 'title' in df.columns:
    df['title_length']     = df['title'].fillna('').str.len()
    df['title_word_count'] = df['title'].fillna('').str.split().str.len()
if 'description' in df.columns:
    df['desc_length']      = df['description'].fillna('').str.len()
    df['desc_word_count']  = df['description'].fillna('').str.split().str.len()

text_feat = [c for c in ['title_length', 'title_word_count', 'desc_length', 'desc_word_count']
             if c in df.columns]
if text_feat:
    print('=== TEXT FEATURE DESCRIPTIVE STATISTICS ===')
    print(df[text_feat].describe().round(2))

# Description length by Job Level
if TARGET in df.columns and 'desc_length' in df.columns:
    plot_df = df[[TARGET, 'desc_length']].dropna()
    order   = plot_df.groupby(TARGET)['desc_length'].median().sort_values(ascending=False).index

    fig, ax = plt.subplots(figsize=(12, 5))
    sns.boxplot(data=plot_df, x=TARGET, y='desc_length',
                order=order, palette='Set2', ax=ax)
    ax.set_title('Job Description Character Length by Experience Level', fontsize=12, fontweight='bold')
    ax.tick_params(axis='x', rotation=30)
    plt.tight_layout()
    plt.show()

**CELL 23: Top 25 Job Description Keywords**

In [ ]:
if 'description' in df.columns:
    stop_words = {'the','and','or','in','of','to','a','for','is','are','with','on',
                  'at','be','as','an','we','our','you','will','this','that','have',
                  'it','from','by','was','not','your','can','has','all','they','their',
                  'work','role','team','job','experience','position','skills','ability'}

    all_words = (df['description'].fillna('')
                   .str.lower()
                   .str.replace(r'[^a-z\s]', ' ', regex=True)
                   .str.split().explode())
    top_keywords = (all_words[(all_words.str.len() > 3) & (~all_words.isin(stop_words))]
                      .value_counts().head(25))

    kw_labels = [str(x) for x in top_keywords.index]
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.bar(kw_labels, top_keywords.values,
           color=sns.color_palette('viridis', 25), edgecolor='white')
    ax.set_title('Top 25 Job Description Keywords (Filtered Stop Words)',
                 fontsize=12, fontweight='bold')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()

## 12. BUSINESS INSIGHTS & EXECUTIVE DASHBOARD
Synthesize analytical insights and provide actionable recommendations for downstream tasks.

---
**CELL 24: Business Insights & Downstream Team Handover Checklist**

In [ ]:
print('=' * 70)
print('         EXECUTIVE BUSINESS INSIGHTS — DS JOB RECOMMEND')
print('=' * 70)

if TARGET in df.columns:
    counts = df[TARGET].value_counts()
    print('\n1. Market Demand Distribution:')
    print(f'   • Most Recruited Level : {counts.idxmax()} ({counts.max():,} postings, {counts.max()/len(df)*100:.1f}%)')
    print(f'   • Least Recruited Level: {counts.idxmin()} ({counts.min():,} postings, {counts.min()/len(df)*100:.1f}%)')

sal_col = next((c for c in ['normalized_salary', 'med_salary'] if c in df.columns), None)
if sal_col and TARGET in df.columns:
    by_level = df.groupby(TARGET)[sal_col].median().sort_values(ascending=False)
    print(f'\n2. Median Salary by Job Level ({sal_col}):')
    for level, val in by_level.items():
        print(f'   • {level:<18}: ${val:,.0f}')

if 'corr_df' in dir() and len(corr_df):
    multi = corr_df[corr_df['Pearson r'].abs() > 0.7]
    print('\n3. Multicollinearity Status:')
    if len(multi):
        for _, row in multi.iterrows():
            print(f'   • {row["Feature 1"]} <-> {row["Feature 2"]}: r = {row["Pearson r"]}')
        print('   -> Recommendation: Retain only normalized_salary for downstream modeling.')

print('\n' + '-' * 70)
print('DOWNSTREAM TEAM HANDOVER CHECKLIST:')
print('→ FOR PERSON 3 (Feature Engineering):')
print('  1. Apply Log-transformation to normalized_salary to mitigate skewness.')
print('  2. Engineer text length features (desc_length, title_word_count).')
print('  3. Encode high-frequency skill tags (IT, SALE, MGMT, ENG) into multi-hot features.')
print('→ FOR PERSON 4 (Modeling):')
print('  1. Mitigate severe class imbalance (34:1) via Stratified K-Fold and class_weight / SMOTE.')
print('  2. Benchmark models using F1-Macro / Weighted F1 rather than raw Accuracy.')
print('=' * 70)

**CELL 25: Executive Summary Dashboard**

In [ ]:
sal_col = next((c for c in ['normalized_salary', 'med_salary'] if c in df.columns), None)

if sal_col and TARGET in df.columns:
    fig, axes = plt.subplots(2, 2, figsize=(18, 12))
    palette = sns.color_palette('Set2')

    # 1. Job Level Distribution
    valid_target = df[TARGET].dropna().value_counts()
    target_x = [str(x) for x in valid_target.index]
    axes[0, 0].bar(target_x, valid_target.values,
                   color=palette[:len(valid_target)], edgecolor='white')
    for i, (cnt, pct) in enumerate(zip(valid_target.values, valid_target / valid_target.sum() * 100)):
        axes[0, 0].text(i, cnt + valid_target.max() * 0.01,
                        f'{pct:.0f}%', ha='center', fontsize=9)
    axes[0, 0].set_title('1. Experience Level Distribution (Job Level)', fontweight='bold')
    axes[0, 0].tick_params(axis='x', rotation=30)

    # 2. Salary by Job Level
    order = df.groupby(TARGET)[sal_col].median().sort_values(ascending=False).index
    sns.boxplot(data=df, x=TARGET, y=sal_col,
                order=order, palette='Set2', ax=axes[0, 1])
    axes[0, 1].set_title(f'2. Salary Distribution by Experience Level ({sal_col})', fontweight='bold')
    axes[0, 1].tick_params(axis='x', rotation=30)

    # 3. Salary Correlation Heatmap
    sal_cols = [c for c in ['min_salary', 'med_salary', 'max_salary', 'normalized_salary']
                if c in df.columns]
    if len(sal_cols) >= 2:
        sns.heatmap(df[sal_cols].corr(), annot=True, fmt='.2f', cmap='RdYlGn',
                    center=0, ax=axes[1, 0], linewidths=0.5, annot_kws={'size': 10})
        axes[1, 0].set_title('3. Salary Feature Multicollinearity Matrix', fontweight='bold')

    # 4. Views vs Applies Scatter Plot
    if 'views' in df.columns and 'applies' in df.columns:
        va_df = df[['views', 'applies']].dropna()
        if len(va_df) > 0:
            sample = va_df.sample(min(3000, len(va_df)), random_state=42)
            axes[1, 1].scatter(sample['views'], sample['applies'],
                               alpha=0.3, s=10, color='steelblue')
            r = va_df.corr().iloc[0, 1]
            axes[1, 1].set_title(f'4. Engagement Relationship: Views vs Applies (r = {r:.2f})', fontweight='bold')
            axes[1, 1].set_xlabel('Views')
            axes[1, 1].set_ylabel('Applies')

    plt.suptitle('EXECUTIVE SUMMARY DASHBOARD — DS JOB RECOMMEND',
                 fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

print('\nExploratory Data Analysis (EDA) completed successfully!')